In [ ]:
import numpy as np, pandas as pd, json
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
df=pd.read_csv('../data/heart.csv')
y=df['target']
x=df.drop(columns=['target'])
num=x.select_dtypes(include=[int,float]).columns.tolist()
cat=[c for c in x.columns if c not in num]
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
est=RandomForestClassifier(class_weight='balanced',random_state=42)
pipe=Pipeline([('pre',pre),('clf',est)])
param_grid={'clf__n_estimators':[200,400],'clf__max_depth':[None,10,20],'clf__min_samples_split':[2,5]}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
g=GridSearchCV(pipe,param_grid=param_grid,cv=cv,scoring='roc_auc',n_jobs=-1).fit(x,y)
pd.DataFrame(g.cv_results_).to_csv('../results/gridsearch_rf.csv',index=False)
open('../models/grid_best.json','w',encoding='utf-8').write(json.dumps({'best_score':float(g.best_score_),'best_params':g.best_params_},ensure_ascii=False,indent=2))